# WatermarkingForLLM on Google Colab

Clones this **public** repo, builds **CPRF** (Go) and **PRC** (Rust/maturin), installs Python deps, and runs **`app.py`**.

**Optional:** Put a **`.env`** in the repo root on the VM with **`HF_TOKEN=`** so §5 can log in non-interactively; otherwise use **`notebook_login()`** there.

**Before you start:** **Runtime → Change runtime type → GPU**. Use **Python 3.11+** if offered.


In [2]:
import sys

assert sys.version_info >= (3, 11), "Use Python 3.11+ (Runtime → Change runtime type)"

import torch

print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()

Python: 3.12.13
CUDA: True — NVIDIA L4


## 1. Clone the repo

Edit **`REPO_OWNER`** / **`REPO_NAME`** in the next cell if you use a fork. The tree is cloned to **`/content/<REPO_NAME>`**.

If you add **`.env`** on the VM (e.g. for Hugging Face), §1 loads it automatically after cloning.


In [3]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# --- edit if you use a fork ---
REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"

PROJECT_ROOT = Path("/content") / REPO_NAME
CLONE_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(
        ["git", "clone", "--depth", "1", CLONE_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    print("Repo already present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
_env = PROJECT_ROOT / ".env"
if _env.is_file():
    try:
        from dotenv import load_dotenv
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
            check=True,
        )
        from dotenv import load_dotenv

    load_dotenv(_env)
    print("Loaded", _env)

print("cwd:", PROJECT_ROOT.resolve())

# --- Hugging Face hub id for the watermark causal LM (edit here; used in sections 7 and 8) ---
WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"


Repo already present: /content/attribute-based-watermarking
cwd: /content/attribute-based-watermarking


## 2. Build CPRF (`cprf.so`)

Linux `.so` only. The next cell aligns `go.mod` to **go 1.23** when needed.
**Go installs from https://go.dev/dl** (official tarball → `/usr/local/go`) when `go` is missing — avoids `apt-get update` hangs on Colab. `apt-get` runs only if the download fails (with timeouts). If nothing works: **Runtime → Restart session** and retry.


In [4]:
import os
import platform
import shutil
import subprocess
from pathlib import Path

cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"

# Colab's apt golang is often older than the repo's `go` directive; relax to 1.23 for the build.
_go_mod = cprf_dir / "go.mod"
if _go_mod.is_file():
    _lines = _go_mod.read_text(encoding="utf-8").splitlines(keepends=True)
    _out, _changed = [], False
    for _line in _lines:
        _s = _line.strip()
        if _s.startswith("go ") and not _s.startswith("go 1.23"):
            _out.append("go 1.23\n")
            _changed = True
        else:
            _out.append(_line)
    if _changed:
        _go_mod.write_text("".join(_out), encoding="utf-8")


def _prepend_path(bin_dir: str) -> None:
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")


def _ensure_go(timeout_s: int = 420) -> None:
    """Prefer official Go tarball (avoids apt hangs on Colab); bounded apt as fallback."""
    if shutil.which("go") is not None:
        return
    env = dict(os.environ)
    env.setdefault("DEBIAN_FRONTEND", "noninteractive")
    sub = dict(check=True, stdin=subprocess.DEVNULL, env=env, timeout=timeout_s)

    prefix = Path("/usr/local")
    goroot = prefix / "go"
    bindir = str(goroot / "bin")
    go_exe = goroot / "bin" / "go"
    if go_exe.is_file():
        _prepend_path(bindir)
        subprocess.run(
            [str(go_exe), "version"],
            check=True,
            stdin=subprocess.DEVNULL,
            timeout=120,
        )
        return

    GO_VER = "1.23.4"
    uarch_map = {"x86_64": "amd64", "AMD64": "amd64", "aarch64": "arm64", "arm64": "arm64"}
    arch = uarch_map.get(platform.machine(), "amd64")
    tgz_name = f"go{GO_VER}.linux-{arch}.tar.gz"
    tgz_path = Path("/tmp") / tgz_name
    dl_url = f"https://go.dev/dl/{tgz_name}"

    print("go missing: fetching", dl_url, "(tarball — avoids slow apt)\n", flush=True)
    curl_kw = dict(check=False, stdin=subprocess.DEVNULL, env=env, timeout=timeout_s)
    dl_ok = False
    if shutil.which("curl"):
        r = subprocess.run(
            [
                "curl", "-fL", "--retry", "3", "--connect-timeout", "30",
                "--max-time", str(timeout_s), "-o", str(tgz_path), dl_url,
            ],
            **curl_kw,
        )
        dl_ok = r.returncode == 0 and tgz_path.is_file() and tgz_path.stat().st_size > 1024 * 1024
    elif shutil.which("wget"):
        r = subprocess.run(
            ["wget", "-nv", "--timeout=120", "--tries=3", "-O", str(tgz_path), dl_url],
            **curl_kw,
        )
        dl_ok = r.returncode == 0 and tgz_path.is_file() and tgz_path.stat().st_size > 1024 * 1024

    if dl_ok:
        if goroot.is_dir():
            shutil.rmtree(goroot)
        subprocess.run(["tar", "-C", str(prefix), "-xzf", str(tgz_path)], **sub)
        _prepend_path(bindir)
        if not go_exe.is_file():
            raise FileNotFoundError(f"Tarball unpacked but missing {go_exe}")
        subprocess.run(
            [str(go_exe), "version"],
            check=True,
            stdin=subprocess.DEVNULL,
            timeout=120,
        )
        return

    print("Tarball failed; trying apt-get with timeout…\n", flush=True)
    try:
        subprocess.run(
            ["apt-get", "update", "-qq", "-o", "Acquire::Retries=3"],
            check=True,
            stdin=subprocess.DEVNULL,
            env=env,
            timeout=timeout_s,
        )
        subprocess.run(
            [
                "apt-get",
                "install",
                "-y",
                "-qq",
                "-o",
                "Dpkg::Use-Pty=0",
                "-o",
                "Acquire::Retries=3",
                "--no-install-recommends",
                "golang-go",
            ],
            check=True,
            stdin=subprocess.DEVNULL,
            env=env,
            timeout=timeout_s,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            "apt-get exceeded timeout — restart Runtime, or install golang-go in a shell, then re-run."
        ) from None


_ensure_go()

_go_exe = shutil.which("go")
if _go_exe is None and (Path("/usr/local/go/bin/go")).is_file():
    _go_exe = "/usr/local/go/bin/go"
if _go_exe is None:
    raise FileNotFoundError("go executable not found after _ensure_go(); check tarball unpack.")

if not so_path.is_file():
    subprocess.run(
        [_go_exe, "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )

assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)


CPRF: /content/attribute-based-watermarking/cprf/cprf.so


## 3. Build PRC (Rust + maturin)

Installs Rust in the VM home if `rustc` is missing, then **`maturin build`** (release wheel) and **`pip install`** that wheel. This avoids `maturin develop` issues on hosted Colab; build stdout/stderr are printed when something fails.

In [5]:
import os
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

try:
    print("Building Rust package with maturin build...")
    completed_process = subprocess.run(
        ["maturin", "build", "--release", "-m", "prc/Cargo.toml"],
        cwd=PROJECT_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )

    if completed_process.stdout:
        print("Maturin build stdout:\n", completed_process.stdout)
    if completed_process.stderr:
        print("Maturin build stderr:\n", completed_process.stderr)

    if completed_process.returncode != 0:
        raise subprocess.CalledProcessError(
            completed_process.returncode,
            completed_process.args,
            output=completed_process.stdout,
            stderr=completed_process.stderr,
        )

    wheel_dir = PROJECT_ROOT / "prc" / "target" / "wheels"
    wheel_files = sorted(
        wheel_dir.glob("*.whl"), key=lambda p: p.stat().st_mtime, reverse=True
    )
    wheel_file = wheel_files[0] if wheel_files else None

    if wheel_file:
        print(f"Installing {wheel_file.name} with pip...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", str(wheel_file)],
            check=True,
        )
    else:
        raise FileNotFoundError("No wheel file found after maturin build.")

except subprocess.CalledProcessError as e:
    print("Maturin failed with error:", e)
    print("Standard Output:", e.stdout)
    print("Standard Error:", e.stderr)
    raise
except FileNotFoundError as e:
    print(e)
    raise

print("PRC: maturin build and pip install ok")

Building Rust package with maturin build...
Maturin build stderr:
 🔗 Found pyo3 bindings
🐍 Found CPython 3.12 at /usr/bin/python3
📡 Using build options features from pyproject.toml
    Finished `release` profile [optimized] target(s) in 0.03s
📦 Built wheel for CPython 3.12 to /content/attribute-based-watermarking/prc/target/wheels/prc-0.1.0-cp312-cp312-manylinux_2_34_x86_64.whl

Installing prc-0.1.0-cp312-cp312-manylinux_2_34_x86_64.whl with pip...
PRC: maturin build and pip install ok


## 4. Python dependencies (Colab `%pip`)

Uses Colab’s recommended install path. Colab usually ships **PyTorch with CUDA**; extra packages match `pyproject.toml` (without the local-only CUDA index).

In [6]:
%pip install -q "transformers==5.5.4" "rich>=15" "keybert>=0.9" sentencepiece accelerate python-dotenv

## 5. Hugging Face login (gated Llama)

Accept the **Llama 3.2** license on the model card.

If **`HF_TOKEN`** or **`HUGGING_FACE_HUB_TOKEN`** is set (environment or **`PROJECT_ROOT/.env`** loaded in §1 or below), you are logged in non-interactively. Otherwise the cell uses **`notebook_login()`**.


In [7]:
import os
from pathlib import Path

from huggingface_hub import login, notebook_login

try:
    _root = PROJECT_ROOT
except NameError:
    _root = Path("/content") / "attribute-based-watermarking"

_env = _root / ".env"
if _env.is_file():
    try:
        from dotenv import load_dotenv

        load_dotenv(_env)
    except ImportError:
        pass

_token = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
)
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF: logged in from environment token.")
else:
    notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 6. Pull latest from GitHub

Run after §1 when you want the newest commit (**public** remote, no token).

Re-run CPRF (§2) / PRC (§3) only if those directories changed or builds fail after an update.


In [8]:
import subprocess
from pathlib import Path

REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"
PROJECT_ROOT = Path("/content") / REPO_NAME

if not (PROJECT_ROOT / ".git").is_dir():
    raise FileNotFoundError("Run §1 first.")

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
log = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout.strip())


49d3166 custom model choice


## 7. Run the demo

Set **`WATERMARK_LLM_ID`** in **section 1** (clone/setup cell). The next cell imports **`watermarking`** and calls **`watermarking.set_llm_model_id(WATERMARK_LLM_ID)`** before **`runpy.run_path(app.py)`** so the notebook picks the LM without environment variables.

To override only for this demo, assign **`WATERMARK_LLM_ID`** again at the top of the next cell before **`import watermarking`**.

Also loads **DeBERTa-v3-large** NLI for ``derive_x``. First run downloads both; VRAM use can be high on a T4.


In [ ]:
import os
import runpy
import sys

# Requires ``PROJECT_ROOT`` from the setup cell (section 1).
try:
    PROJECT_ROOT
except NameError as e:
    raise RuntimeError(
        "PROJECT_ROOT is not defined; run the repo setup cell (section 1) first."
    ) from e

os.chdir(PROJECT_ROOT)

# Optional: override the hub id from section 1 for this cell only.
WATERMARK_LLM_ID = "Qwen/Qwen2.5-3B-Instruct"

try:
    _llm_demo = WATERMARK_LLM_ID
except NameError as e:
    raise RuntimeError(
        "WATERMARK_LLM_ID is not defined; set it in section 1 (clone/setup cell) before running this cell."
    ) from e

import watermarking as wm

wm.set_llm_model_id(_llm_demo)

# ``app.py`` ends with ``raise SystemExit(main())``. Jupyter shows that as an exception
# traceback even when the script only returns a nonzero exit code (failed checks).
try:
    runpy.run_path(str(PROJECT_ROOT / "app.py"), run_name="__main__")
except SystemExit as exc:
    code = exc.code
    if code not in (0, None):
        print("app.py exited with code", repr(code), "(0 means all protocol checks passed).")


Loading causal LM 'Qwen/Qwen2.5-7B-Instruct' on cuda


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

──────────────────────────────────────────────── Wrong transcript ─────────────────────────────────────────────────

(truncated to 400 chars)
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text u

master_detect → False  recovered: 0010111000111010001000011111000100111000000100000111111001111010... (300 total)

PASS  master_detect(wrong transcript) should be False  (got False, expect False)

╭─────────────────────────────╮
│ All protocol checks passed. │
╰─────────────────────────────╯

## 8. Run the policy benchmark

Same in-process style as **section 7** (`runpy.run_path`): no subprocess, so Rich output goes straight to the cell like `app.py`. Edit the `BENCHMARK_*` constants in the next cell; `sys.argv` is set so the script's `argparse` matches the CLI.

**Prompt list:** `BENCHMARK_PROMPT_CASES` is a list of strings, each `"id:prompt"` (only the first `:` splits id from prompt). If **non-empty**, the benchmark runs **only** those cases. If **empty**, the script uses built-in defaults (`benchmark_policy_detection.DEFAULT_PROMPT_CASES`).

**Causal LM:** the next cell calls **`watermarking.set_llm_model_id`** with **`WATERMARK_LLM_ID`** from **section 1** (redefine that name in the cell if you skipped section 1).

Assume **section 1** defined `PROJECT_ROOT` and `WATERMARK_LLM_ID`. Run **section 7** first if you want warm HF caches; the benchmark loads the same models again.


In [ ]:
import os
import runpy
import sys

# --- edit these, then run the cell (same pattern as the app.py demo in section 7) ---
BENCHMARK_RUNS = 1
BENCHMARK_CODE_LENGTH = 600
BENCHMARK_MODULUS = 1024
BENCHMARK_REUSE_KEY = False  # True -> add --reuse-key (one CPRF key per prompt id across runs)

# Full prompt list: each entry "id:prompt" (first ':' only splits id from prompt).
# Non-empty -> benchmark uses ONLY these cases. Empty -> script defaults (dcf_finance, med_school).
BENCHMARK_PROMPT_CASES: list[str] = [
    # "sports:Summarize the rules of pickleball for a beginner.",
    # "code:Explain what a Python list comprehension is.",
]

os.chdir(PROJECT_ROOT)

_llm_id = "Qwen/Qwen2.5-3B-Instruct"
if "WATERMARK_LLM_ID" in globals():
    _llm_id = WATERMARK_LLM_ID

import watermarking as wm

wm.set_llm_model_id(_llm_id)

_bench_argv = sys.argv
sys.argv = [
    str(PROJECT_ROOT / "benchmark_policy_detection.py"),
    "--runs",
    str(BENCHMARK_RUNS),
    "--code-length",
    str(BENCHMARK_CODE_LENGTH),
    "--modulus",
    str(BENCHMARK_MODULUS),
]
if BENCHMARK_REUSE_KEY:
    sys.argv.append("--reuse-key")
for spec in BENCHMARK_PROMPT_CASES:
    sys.argv.extend(["--prompt-case", spec])

try:
    runpy.run_path(str(PROJECT_ROOT / "benchmark_policy_detection.py"), run_name="__main__")
finally:
    sys.argv = _bench_argv


code_length=600  modulus=1024  runs=1  keys=fresh per run  llm='Qwen/Qwen2.5-7B-Instruct'